In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")

print("Path to dataset files:", path)

/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/former_names.csv
/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/goalscorers.csv
/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/shootouts.csv
/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/results.csv
Path to dataset files: /kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017


In [31]:
#remove all unplayed games
alldata = pd.read_csv('/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/results.csv')
playedmatches = alldata[alldata['home_score'].notnull()].copy()

In [32]:
#step 1 - calculate average goals scored and conceded for each team, ,home and away, since 2010
playedmatches['date'] = pd.to_datetime(playedmatches['date'])
recent = playedmatches[playedmatches['date'] >= '2010-01-01']
#gs is 'goals scored', gc is 'goals conceded'

#for home
homeview = recent[['home_team', 'home_score', 'away_score']]
homeview = homeview.rename(columns = {'home_team' : 'team', 'home_score' : 'gs', 'away_score' : 'gc'})

#away
awayview = recent[['away_team', 'away_score', 'home_score']]
awayview = awayview.rename(columns = {'away_team':'team', 'away_score':'gs', 'home_score':'gc'})

#combine the two dataframes for grouping
combined = pd.concat([homeview, awayview])

#groupby
grouped = combined.groupby('team')[['gs', 'gc']].mean()

#set baselines for all goals scored and conceded
avgs = combined['gs'].mean()
avgc = combined['gc'].mean()

#calculate team strength and defence as a multiple of baselines
grouped['attack'] = grouped['gs'] / avgs
grouped['defence'] = grouped['gc'] / avgc

#remove junk (less than 50 matches played)
counts = combined.groupby('team').size()
grouped['matches'] = counts
#"big teams only"
bigsorted = (grouped[grouped['matches'] >= 50]).sort_values('attack', ascending = False)
bigsorted





,gs,gc,attack,defence,matches
team,,,,,
New Caledonia,2.443038,1.215190,1.790448,0.890585,79
Germany,2.342593,1.092593,1.716833,0.800737,216
Spain,2.302752,0.738532,1.687635,0.541254,218
Belgium,2.266667,0.902564,1.661189,0.661469,195
Netherlands,2.222772,0.995050,1.629020,0.729249,202
...,...,...,...,...,...
Bhutan,0.524590,3.688525,0.384460,2.703237,61
Anguilla,0.482143,3.500000,0.353352,2.565071,56
Liechtenstein,0.371429,2.592857,0.272212,1.900247,140
